In [1]:
import pandas as pd
import numpy as np

from catboost import CatBoostRegressor

In [2]:
path = 'processed/v1/train_wide_cleaned.csv'
output_path = 'processed/v2/train_wide_cleaned_with_income.csv'

df = pd.read_csv(path)

df.head()

,user_id,sex,age,is_new_customer,seniority_months,region_code,region_name,income,segment,dep-7,...,dep-2,loan-2,card-4,loan-1,srv-3,loan-5,biz-4,dep-6,loan-4,card-3
0,657788,F,42.0,0.0,114.0,28.0,MADRID,132559.35,INDIVIDUALS,0,...,0,0,0,0,0,0,0,0,0,0
1,657795,M,44.0,0.0,114.0,26.0,"RIOJA, LA",81399.57,INDIVIDUALS,0,...,0,0,0,0,0,0,0,0,0,0
2,657790,M,42.0,0.0,114.0,48.0,BIZKAIA,NaN,INDIVIDUALS,0,...,0,0,0,0,0,1,0,0,1,1
3,657794,F,49.0,0.0,114.0,8.0,BARCELONA,102189.00,VIP,0,...,0,0,0,0,0,0,0,0,0,0
4,657789,M,36.0,0.0,91.0,28.0,MADRID,153725.49,VIP,0,...,0,0,0,0,0,1,0,0,0,0


In [3]:
RANDOM_SEED = 52
rng = np.random.default_rng(RANDOM_SEED)

In [4]:
def get_result_cols(path):
    df = pd.read_csv(path)

    cols = df.columns.tolist()

    cols.remove("income")

    cols.append("income_generated")
    cols.append("income_filled")


    return cols

In [5]:
def make_age_group(age):
    if age < 18:
        return 'UNDER_18'
    if age <= 24:
        return '18_24'
    if age <= 34:
        return '25_34'
    if age <= 44:
        return '35_44'
    if age <= 54:
        return '45_54'
    if age <= 64:
        return '55_64'
    return '65_PLUS'

In [6]:
def prepare_income_features(df):
    df = df.copy()

    PREFIXES = ["dep-", "card-", "rko-", "loan-", "srv-", "biz-"]

    product_cols = [
        col for col in df.columns
        if col.startswith(tuple(PREFIXES))
    ]

    # create age groups
    df["age_group"] = df["age"].apply(make_age_group)

    # product flags
    for col in product_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).astype("float64")

    dep_cols = [col for col in product_cols if col.startswith("dep-")]
    card_cols = [col for col in product_cols if col.startswith("card-")]
    rko_cols = [col for col in product_cols if col.startswith("rko-")]
    loan_cols = [col for col in product_cols if col.startswith("loan-")]
    srv_cols = [col for col in product_cols if col.startswith("srv-")]
    biz_cols = [col for col in product_cols if col.startswith("biz-")]

    # aggregate product features
    df["products_count"] = df[product_cols].sum(axis=1) if product_cols else 0.0

    df["dep_count"] = df[dep_cols].sum(axis=1) if dep_cols else 0.0
    df["card_count"] = df[card_cols].sum(axis=1) if card_cols else 0.0
    df["rko_count"] = df[rko_cols].sum(axis=1) if rko_cols else 0.0
    df["loan_count"] = df[loan_cols].sum(axis=1) if loan_cols else 0.0
    df["srv_count"] = df[srv_cols].sum(axis=1) if srv_cols else 0.0
    df["biz_count"] = df[biz_cols].sum(axis=1) if biz_cols else 0.0

    df["has_dep"] = (df["dep_count"] > 0).astype(int)
    df["has_card"] = (df["card_count"] > 0).astype(int)
    df["has_rko"] = (df["rko_count"] > 0).astype(int)
    df["has_loan"] = (df["loan_count"] > 0).astype(int)
    df["has_srv"] = (df["srv_count"] > 0).astype(int)
    df["has_biz"] = (df["biz_count"] > 0).astype(int)

    # create product profile
    def product_profile(row):
        if row["products_count"] == 0:
            return "NO_PRODUCTS"

        profile = []

        if row["has_dep"]:
            profile.append("DEP")
        if row["has_card"]:
            profile.append("CARD")
        if row["has_rko"]:
            profile.append("RKO")
        if row["has_loan"]:
            profile.append("LOAN")
        if row["has_srv"]:
            profile.append("SRV")
        if row["has_biz"]:
            profile.append("BIZ")

        return "_".join(profile)

    df["product_profile"] = df.apply(product_profile, axis=1)

    return df, product_cols

In [7]:
def generate_income_from_existing_data(path):
    df = pd.read_csv(path, low_memory=False)

    # prepare features
    df, product_cols = prepare_income_features(df)

    train_mask = df["income"].notna() & (df["income"] > 0)
    train_df = df.loc[train_mask].copy()

    # emissions limit
    lower = train_df["income"].quantile(0.005)
    upper = train_df["income"].quantile(0.900)

    train_df = train_df[
        (train_df["income"] >= lower) &
        (train_df["income"] <= upper)
    ].copy()

    y_train = train_df["income"]

    # categorical features
    cat_features = [
        # base features
        "sex",
        "segment",
        "age_group",
        "product_profile",

        # region features
        "region_code",
        "region_name"
    ]

    # numerical features
    num_features = [
        "age",
        "is_new_customer",
        "seniority_months",
        "products_count",
        "dep_count",
        "card_count",
        "rko_count",
        "loan_count",
        "srv_count",
        "biz_count",
        "has_dep",
        "has_card",
        "has_rko",
        "has_loan",
        "has_srv",
        "has_biz"
    ]

    # product flags
    num_features = num_features + product_cols

    features = cat_features + num_features

    for col in cat_features:
        df[col] = df[col].fillna("UNKNOWN").astype(str)
        train_df[col] = train_df[col].fillna("UNKNOWN").astype(str)

    for col in num_features:
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).astype("float64")
        train_df[col] = pd.to_numeric(train_df[col], errors="coerce").fillna(0).astype("float64")

    X_train = train_df[features]
    X_all = df[features]

    model = CatBoostRegressor(
        loss_function="MAE",
        eval_metric="MAE",
        iterations=1500,
        learning_rate=0.04,
        depth=7,
        l2_leaf_reg=7,
        random_seed=RANDOM_SEED,
        verbose=100,
        allow_writing_files=False
    )

    model.fit(
        X_train,
        y_train,
        cat_features=cat_features
    )

    # deterministic prediction
    pred_all = model.predict(X_all)
    pred_all = np.clip(pred_all, lower, upper)

    df["income_model_prediction"] = np.round(pred_all, 2)

    missing_mask = df["income"].isna() | (df["income"] <= 0)

    # prediction for every row
    df["income_generated"] = df["income_model_prediction"]

    # flag: 1 means original income was missing and was filled by prediction
    missing_mask = df["income"].isna() | (df["income"] <= 0)

    df["income_filled"] = df["income"]

    df.loc[missing_mask, "income_filled"] = df.loc[
        missing_mask,
        "income_generated"
    ]

    df["income_was_generated"] = missing_mask.astype(int)

    # keep result columns
    cols = get_result_cols(path)

    cols = [
        col for col in cols
        if col in df.columns
    ]

    df = df[cols]

    return df, model

In [8]:
df, model = generate_income_from_existing_data(path)

df.to_csv(output_path, index=False)

0:	learn: 40551.8686616	total: 246ms	remaining: 6m 9s
100:	learn: 36143.3302190	total: 7.36s	remaining: 1m 41s
200:	learn: 36025.9182843	total: 14.6s	remaining: 1m 34s
300:	learn: 35962.3029964	total: 21.2s	remaining: 1m 24s
400:	learn: 35895.0794285	total: 28s	remaining: 1m 16s
500:	learn: 35846.0932853	total: 34.7s	remaining: 1m 9s
600:	learn: 35805.6952563	total: 41.4s	remaining: 1m 1s
700:	learn: 35769.1259838	total: 48s	remaining: 54.7s
800:	learn: 35735.5432356	total: 54.6s	remaining: 47.6s
900:	learn: 35706.4810134	total: 1m	remaining: 40.5s
1000:	learn: 35682.6901062	total: 1m 7s	remaining: 33.6s
1100:	learn: 35659.4934739	total: 1m 13s	remaining: 26.8s
1200:	learn: 35637.5765372	total: 1m 20s	remaining: 20s
1300:	learn: 35618.3835280	total: 1m 26s	remaining: 13.3s
1400:	learn: 35600.5183657	total: 1m 33s	remaining: 6.59s
1499:	learn: 35586.2213690	total: 1m 39s	remaining: 0us


In [9]:
# import os
# import pandas as pd
# import numpy as np

# from catboost import CatBoostRegressor

# from sklearn.model_selection import train_test_split
# from sklearn.metrics import (
#     mean_absolute_error,
#     mean_squared_error,
#     median_absolute_error,
#     r2_score
# )


# # =========================
# # CONFIG
# # =========================

# RANDOM_SEED = 52

# RAW_PATH = "processed/v1/train_wide_cleaned.csv"
# OUTPUT_PATH = "processed/v2/train_wide_cleaned_with_income.csv"
# METRICS_PATH = "processed/v2/income_metrics.csv"

# TRAIN_INCOME_LOWER_QUANTILE = 0.005
# TRAIN_INCOME_UPPER_QUANTILE = 0.900


# # =========================
# # METRICS
# # =========================

# def calculate_income_metrics(y_true, y_pred, title="Metrics"):
#     y_true = np.array(y_true)
#     y_pred = np.array(y_pred)

#     y_pred = np.maximum(y_pred, 0)

#     mae = mean_absolute_error(y_true, y_pred)
#     rmse = mean_squared_error(y_true, y_pred) ** 0.5
#     medae = median_absolute_error(y_true, y_pred)
#     mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
#     rmsle = mean_squared_error(
#         np.log1p(y_true),
#         np.log1p(y_pred)
#     ) ** 0.5
#     r2 = r2_score(y_true, y_pred)

#     print(f"\n=== {title} ===")
#     print(f"Rows: {len(y_true):,}")
#     print(f"MAE: {mae:,.2f}")
#     print(f"RMSE: {rmse:,.2f}")
#     print(f"MedianAE: {medae:,.2f}")
#     print(f"MAPE: {mape:.2f}%")
#     print(f"RMSLE: {rmsle:.4f}")
#     print(f"R2: {r2:.4f}")

#     return {
#         "title": title,
#         "rows": len(y_true),
#         "mae": mae,
#         "rmse": rmse,
#         "median_ae": medae,
#         "mape": mape,
#         "rmsle": rmsle,
#         "r2": r2
#     }


# # =========================
# # FEATURE ENGINEERING
# # =========================

# def make_age_group(age):
#     if pd.isna(age):
#         return "UNKNOWN"
#     if age < 18:
#         return "UNDER_18"
#     if age <= 24:
#         return "18_24"
#     if age <= 34:
#         return "25_34"
#     if age <= 44:
#         return "35_44"
#     if age <= 54:
#         return "45_54"
#     if age <= 64:
#         return "55_64"
#     return "65_PLUS"


# def get_result_cols(path):
#     df = pd.read_csv(path, low_memory=False)

#     cols = df.columns.tolist()

#     if "income" in cols:
#         cols.remove("income")

#     for col in [
#         "income_model_prediction",
#         "income_generated",
#         "income_filled",
#         "income_was_generated"
#     ]:
#         if col not in cols:
#             cols.append(col)

#     return cols


# def prepare_income_features(df):
#     df = df.copy()

#     PREFIXES = ["dep-", "card-", "rko-", "loan-", "srv-", "biz-"]

#     product_cols = [
#         col for col in df.columns
#         if col.startswith(tuple(PREFIXES))
#     ]

#     # categorical base features
#     df["sex"] = df["sex"].fillna("UNKNOWN").astype(str)
#     df["segment"] = df["segment"].fillna("UNKNOWN").astype(str)

#     # region features
#     if "region_code" in df.columns:
#         df["region_code"] = df["region_code"].fillna("UNKNOWN").astype(str)
#     else:
#         df["region_code"] = "UNKNOWN"

#     if "region_name" in df.columns:
#         df["region_name"] = df["region_name"].fillna("UNKNOWN").astype(str)
#     else:
#         df["region_name"] = "UNKNOWN"

#     # age
#     df["age"] = pd.to_numeric(df["age"], errors="coerce")
#     df.loc[
#         (df["age"] < 14) | (df["age"] > 95),
#         "age"
#     ] = np.nan
#     df = df.dropna(subset=["age"])
#     df["age"] = df["age"].round().astype("float64")

#     # seniority_months
#     df["seniority_months"] = pd.to_numeric(
#         df["seniority_months"],
#         errors="coerce"
#     )
#     df.loc[df["seniority_months"] < 0, "seniority_months"] = np.nan
#     df = df.dropna(subset=["seniority_months"])
#     df["seniority_months"] = df["seniority_months"].round().astype("float64")

#     # is_new_customer
#     df["is_new_customer"] = pd.to_numeric(
#         df["is_new_customer"],
#         errors="coerce"
#     ).fillna(0).astype("float64")

#     # age group
#     df["age_group"] = df["age"].apply(make_age_group)

#     # product flags
#     for col in product_cols:
#         df[col] = pd.to_numeric(
#             df[col],
#             errors="coerce"
#         ).fillna(0).astype("float64")

#     dep_cols = [col for col in product_cols if col.startswith("dep-")]
#     card_cols = [col for col in product_cols if col.startswith("card-")]
#     rko_cols = [col for col in product_cols if col.startswith("rko-")]
#     loan_cols = [col for col in product_cols if col.startswith("loan-")]
#     srv_cols = [col for col in product_cols if col.startswith("srv-")]
#     biz_cols = [col for col in product_cols if col.startswith("biz-")]

#     df["products_count"] = df[product_cols].sum(axis=1) if product_cols else 0.0

#     df["dep_count"] = df[dep_cols].sum(axis=1) if dep_cols else 0.0
#     df["card_count"] = df[card_cols].sum(axis=1) if card_cols else 0.0
#     df["rko_count"] = df[rko_cols].sum(axis=1) if rko_cols else 0.0
#     df["loan_count"] = df[loan_cols].sum(axis=1) if loan_cols else 0.0
#     df["srv_count"] = df[srv_cols].sum(axis=1) if srv_cols else 0.0
#     df["biz_count"] = df[biz_cols].sum(axis=1) if biz_cols else 0.0

#     df["has_dep"] = (df["dep_count"] > 0).astype(int)
#     df["has_card"] = (df["card_count"] > 0).astype(int)
#     df["has_rko"] = (df["rko_count"] > 0).astype(int)
#     df["has_loan"] = (df["loan_count"] > 0).astype(int)
#     df["has_srv"] = (df["srv_count"] > 0).astype(int)
#     df["has_biz"] = (df["biz_count"] > 0).astype(int)

#     def product_profile(row):
#         if row["products_count"] == 0:
#             return "NO_PRODUCTS"

#         profile = []

#         if row["has_dep"]:
#             profile.append("DEP")
#         if row["has_card"]:
#             profile.append("CARD")
#         if row["has_rko"]:
#             profile.append("RKO")
#         if row["has_loan"]:
#             profile.append("LOAN")
#         if row["has_srv"]:
#             profile.append("SRV")
#         if row["has_biz"]:
#             profile.append("BIZ")

#         return "_".join(profile)

#     df["product_profile"] = df.apply(product_profile, axis=1)

#     return df, product_cols


# # =========================
# # MAIN PIPELINE
# # =========================

# def generate_income_from_existing_data(path):
#     df = pd.read_csv(path, low_memory=False)

#     df, product_cols = prepare_income_features(df)

#     train_mask = df["income"].notna() & (df["income"] > 0)

#     train_df = df.loc[train_mask].copy()

#     lower = train_df["income"].quantile(TRAIN_INCOME_LOWER_QUANTILE)
#     upper = train_df["income"].quantile(TRAIN_INCOME_UPPER_QUANTILE)

#     train_df = train_df[
#         (train_df["income"] >= lower) &
#         (train_df["income"] <= upper)
#     ].copy()

#     # categorical features
#     cat_features = [
#         "sex",
#         "segment",
#         "age_group",
#         "product_profile",

#         "region_code",
#         "region_name"
#     ]

#     # numerical features
#     num_features = [
#         "age",
#         "is_new_customer",
#         "seniority_months",
#         "products_count",
#         "dep_count",
#         "card_count",
#         "rko_count",
#         "loan_count",
#         "srv_count",
#         "biz_count",
#         "has_dep",
#         "has_card",
#         "has_rko",
#         "has_loan",
#         "has_srv",
#         "has_biz"
#     ]

#     num_features = num_features + product_cols

#     features = cat_features + num_features

#     for col in cat_features:
#         df[col] = df[col].fillna("UNKNOWN").astype(str)
#         train_df[col] = train_df[col].fillna("UNKNOWN").astype(str)

#     for col in num_features:
#         df[col] = pd.to_numeric(
#             df[col],
#             errors="coerce"
#         ).fillna(0).astype("float64")

#         train_df[col] = pd.to_numeric(
#             train_df[col],
#             errors="coerce"
#         ).fillna(0).astype("float64")

#     X = train_df[features]
#     y = train_df["income"]

#     segment_counts = train_df["segment"].value_counts()
#     stratify_col = train_df["segment"] if segment_counts.min() >= 2 else None

#     X_train, X_test, y_train, y_test = train_test_split(
#         X,
#         y,
#         test_size=0.2,
#         random_state=RANDOM_SEED,
#         shuffle=True,
#         stratify=stratify_col
#     )

#     X_all = df[features]

#     print("\nTrain rows:", len(X_train))
#     print("Test rows:", len(X_test))

#     model = CatBoostRegressor(
#         loss_function="MAE",
#         eval_metric="MAE",
#         iterations=1500,
#         learning_rate=0.04,
#         depth=7,
#         l2_leaf_reg=7,
#         random_seed=RANDOM_SEED,
#         verbose=100,
#         allow_writing_files=False
#     )

#     model.fit(
#         X_train,
#         y_train,
#         cat_features=cat_features,
#         eval_set=(X_test, y_test),
#         use_best_model=True
#     )

#     train_pred = model.predict(X_train)
#     test_pred = model.predict(X_test)

#     train_pred = np.clip(train_pred, lower, upper)
#     test_pred = np.clip(test_pred, lower, upper)

#     train_metrics = calculate_income_metrics(
#         y_true=y_train,
#         y_pred=train_pred,
#         title="CatBoost base + region on train"
#     )

#     test_metrics = calculate_income_metrics(
#         y_true=y_test,
#         y_pred=test_pred,
#         title="CatBoost base + region on test"
#     )

#     global_median_pred = np.full(len(y_test), y_train.median())

#     median_metrics = calculate_income_metrics(
#         y_true=y_test,
#         y_pred=global_median_pred,
#         title="Global median baseline on test"
#     )

#     metrics = {
#         "train": train_metrics,
#         "test": test_metrics,
#         "global_median_test": median_metrics
#     }

#     metrics_df = pd.DataFrame(metrics).T

#     print("\nMetrics summary:")
#     print(metrics_df)

#     # deterministic prediction for all rows
#     pred_all = model.predict(X_all)
#     pred_all = np.clip(pred_all, lower, upper)

#     df["income_model_prediction"] = np.round(pred_all, 2)

#     missing_mask = df["income"].isna() | (df["income"] <= 0)

#     df["income_generated"] = np.nan
#     df.loc[missing_mask, "income_generated"] = df.loc[
#         missing_mask,
#         "income_model_prediction"
#     ]

#     df["income_filled"] = df["income"]
#     df.loc[missing_mask, "income_filled"] = df.loc[
#         missing_mask,
#         "income_generated"
#     ]

#     df["income_was_generated"] = missing_mask.astype(int)

#     cols = get_result_cols(path)

#     cols = [
#         col for col in cols
#         if col in df.columns
#     ]

#     df = df[cols]

#     return df, model, metrics_df


# def main():
#     path = RAW_PATH

#     df_income, income_model, metrics_df = generate_income_from_existing_data(path)

#     os.makedirs(
#         os.path.dirname(OUTPUT_PATH),
#         exist_ok=True
#     )

#     df_income.to_csv(OUTPUT_PATH, index=False)
#     metrics_df.to_csv(METRICS_PATH, index=True)

#     print("\nSaved dataset:")
#     print(OUTPUT_PATH)

#     print("\nSaved metrics:")
#     print(METRICS_PATH)

#     print("\nGenerated income rows:")
#     print(int(df_income["income_was_generated"].sum()))


# main()